# Argus VLM Optimization — Notebook 06: Delta Captioning

**Goal:** Compare full scene re-captioning vs delta captioning with structured scene state tracking.


In [ ]:
# Cell 1: Install Dependencies
!pip install -q torch transformers accelerate pillow pyyaml pandas matplotlib seaborn opencv-python-headless rouge-score


In [ ]:
# Cell 2: Imports & Environment Check
import os
import sys
from pathlib import Path
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import pandas as pd
from PIL import Image, ImageDraw
from src.frame_optimization.delta_caption import DeltaCaptioner, SceneState
from src.benchmarking.metrics import compute_rouge_l


In [ ]:
# Cell 3: Configuration
recaption_intervals = [5, 10, 20]
scene_cut_threshold = 0.40


In [ ]:
# Cell 4: Model Loading
from src.vlm.qwen_vlm import QwenVLMWrapper
try:
    wrapper = QwenVLMWrapper(model_name="Qwen/Qwen2.5-VL-3B-Instruct")
except Exception as e:
    print(f"VLM loading: {e}")
    wrapper = None


In [ ]:
# Cell 5: Sequential Video Sequence
seq_frames = []
for i in range(15):
    img = Image.new("RGB", (320, 240), color=(90, 90, 90))
    d = ImageDraw.Draw(img)
    if i > 2:
        d.ellipse([100 + i*5, 100, 140 + i*5, 140], fill=(0, 255, 0))
    seq_frames.append(img)


In [ ]:
# Cell 6: Delta vs Full Execution
captioner = DeltaCaptioner(full_recaption_every=5, scene_cut_threshold=scene_cut_threshold)
captioner.reset()

results = []
output_csv = repo_root / "results" / "static_frames" / "delta_caption_results.csv"

for i, f in enumerate(seq_frames):
    diff_score = 0.05 if i > 0 else 1.0
    rec = captioner.caption(frame=f, frame_idx=i, diff_score=diff_score, vlm_wrapper=wrapper)
    results.append(rec.to_dict())

df = pd.DataFrame(results)
df.to_csv(output_csv, index=False)
print(df[["frame_idx", "caption_type", "triggered_by", "output_token_count"]])


In [ ]:
# Cell 7: Summary Analysis
print("Full captions count:", (df["caption_type"] == "full").sum())
print("Delta captions count:", (df["caption_type"] == "delta").sum())
